# Statistics Practical — Short & Simple Code
Experiments: **1, 2, 3, 4, 6, 8, 9**.

Run each experiment's code cell separately, from top to bottom. Experiment 1, 6, 8 and 9 use built-in datasets. Experiments 2 and 3 use `titanic.csv` from your Desktop. Experiment 4 uses a small demo dataset; use your college CSV if your teacher provided one.


## Experiment 1: Exploratory Data Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

df = load_iris(as_frame=True).frame
print(df.describe())
print("\nMissing values:\n", df.isnull().sum())
print("\nCorrelation:\n", df.corr())

df.hist(figsize=(8, 6))
plt.tight_layout()
plt.show()

sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

plt.hist(df["sepal length (cm)"], edgecolor="black")
plt.title("Sepal Length Distribution")
plt.show()


## Experiment 2: Data Preprocessing

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(r"C:\Users\HITEN PAHILAJANI\Desktop\titanic.csv")
print("Missing before:\\n", df.isnull().sum())

for c in df.select_dtypes(include="number"):
    df[c] = df[c].fillna(df[c].median())
for c in df.select_dtypes(include="object"):
    df[c] = df[c].fillna(df[c].mode()[0])

num = df.select_dtypes(include="number")
df[num.columns] = StandardScaler().fit_transform(num)
df = pd.get_dummies(df, drop_first=True)
print("\\nMissing after:", df.isnull().sum().sum())
print(df.head())


## Experiment 3: Sampling Techniques

In [ ]:
import pandas as pd

df = pd.read_csv(r"C:\Users\HITEN PAHILAJANI\Desktop\titanic.csv")
print("Simple random sample:\\n", df.sample(5, random_state=1))
print("\\nSystematic sample:\\n", df.iloc[::10].head())

if "Pclass" in df.columns:
    print("\\nStratified sample:\\n",
          df.groupby("Pclass", group_keys=False).sample(frac=0.2, random_state=1))
if "Embarked" in df.columns:
    group = df["Embarked"].dropna().unique()[0]
    print("\\nCluster sample:\\n", df[df["Embarked"] == group].head())


## Experiment 4: Hypothesis Testing

In [ ]:
import pandas as pd
from scipy import stats

# Demo data; replace with your college student-performance data if available
df = pd.DataFrame({
    "Score": [65,70,72,68,75,80,77,73,69,74,82,78],
    "Method": ["A"]*6 + ["B"]*6,
    "Gender": ["F","M"]*6,
    "Result": ["Pass","Pass","Pass","Pass","Fail","Pass",
               "Pass","Pass","Fail","Pass","Pass","Pass"]
})

print("One-sample t-test:", stats.ttest_1samp(df["Score"], 70))
print("Two-sample t-test:", stats.ttest_ind(
    df[df.Method=="A"]["Score"], df[df.Method=="B"]["Score"]))
print("Chi-square:", stats.chi2_contingency(pd.crosstab(df["Gender"], df["Result"])))
print("ANOVA:", stats.f_oneway(
    df[df.Method=="A"]["Score"], df[df.Method=="B"]["Score"]))


## Experiment 6: Linear Regression and Diagnostics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=1)
model = LinearRegression().fit(Xtr, ytr)
pred = model.predict(Xte)
print("R2:", r2_score(yte, pred))
print("MSE:", mean_squared_error(yte, pred))

plt.scatter(pred, yte-pred)
plt.axhline(0, color="red")
plt.xlabel("Predicted"); plt.ylabel("Residual")
plt.title("Residual Plot"); plt.show()

Xv = sm.add_constant(X)
print(pd.DataFrame({"Feature": Xv.columns,
    "VIF": [variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1])]}))


## Experiment 8: Classification Models and ROC

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc

data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.2,
                                      random_state=1, stratify=data.target)
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Naive Bayes": GaussianNB()
}
for name, model in models.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    prob = model.predict_proba(Xte)[:, 1]
    print("\n", name, "Accuracy:", accuracy_score(yte, pred))
    print(classification_report(yte, pred))
    fpr, tpr, _ = roc_curve(yte, prob)
    plt.plot(fpr, tpr, label=f"{name} AUC={auc(fpr,tpr):.2f}")

plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves"); plt.legend(); plt.show()


## Experiment 9: Decision Tree and Random Forest

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

data = load_iris()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.2,
                                      random_state=1, stratify=data.target)
models = {"Decision Tree": DecisionTreeClassifier(random_state=1),
          "Random Forest": RandomForestClassifier(random_state=1)}
for name, model in models.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    print("\n", name, "Accuracy:", accuracy_score(yte, pred))
    print(classification_report(yte, pred, target_names=data.target_names))
    print("Confusion matrix:\n", confusion_matrix(yte, pred))
    print("Feature importance:", model.feature_importances_)

plt.figure(figsize=(12,7))
plot_tree(models["Decision Tree"], feature_names=data.feature_names,
          class_names=data.target_names, filled=True)
plt.show()
